In [1]:
import numpy as np 
import pandas as pd

In [2]:
df=pd.read_csv("hotels.csv")

In [3]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

## Brief number checks with Pandas

The next cell reads the CSV and shows the brief values beside data-derived checks. Cancellation rates use train and choose only; the proof-period cancellation labels are deliberately left untouched until the final backtest. Costs and walk limits are client assumptions, not values that can be measured from booking rows.

The charter draft still labels its booking-level decision-unit guess as probably wrong. The contract's action unit is one hotel on one arrival night. Reconcile the charter with the measured checks before treating it as final.

In [4]:
import pandas as pd
from pathlib import Path

# Find the CSV whether this notebook runs from notebooks/, data/, or repo root.
candidates = [Path("hotels.csv"), Path("../data/hotels.csv"), Path("data/hotels.csv")]
csv_path = next(path for path in candidates if path.exists())
bookings = pd.read_csv(csv_path)
raw_rows, raw_columns = bookings.shape

# Build the arrival date from the fields present in the original 32-column file.
arrival_text = (
    bookings["arrival_date_year"].astype(str)
    + "-"
    + bookings["arrival_date_month"]
    + "-"
    + bookings["arrival_date_day_of_month"].astype(str)
)
bookings["arrival_date"] = pd.to_datetime(arrival_text, format="%Y-%B-%d")

train = bookings[bookings["arrival_date"] < "2016-07-01"]
choose = bookings[bookings["arrival_date"].between("2016-07-01", "2016-12-31")]
proof = bookings[bookings["arrival_date"].between("2017-01-01", "2017-08-31")]

# Check cancellation claims only on train and choose. Do not inspect proof labels here.
pre_proof = pd.concat([train, choose])
pre_proof_rate = pre_proof["is_canceled"].mean()
hotel_rates = (
    pre_proof.groupby("hotel")["is_canceled"].mean().mul(100).round(1)
)
hotel_rate_text = "; ".join(
    f"{hotel}: {rate:.1f}%" for hotel, rate in hotel_rates.items()
)

checks = [
    {
        "Brief number or claim": "119,390 bookings; 32 columns",
        "Pandas result": f"{raw_rows:,} rows; {raw_columns} columns",
        "Reading": "Matches the brief" if (raw_rows, raw_columns) == (119390, 32) else "Does not match; investigate",
    },
    {
        "Brief number or claim": "Arrivals July 2015–August 2017",
        "Pandas result": f"{bookings['arrival_date'].min():%Y-%m-%d} to {bookings['arrival_date'].max():%Y-%m-%d}",
        "Reading": "Matches the stated range" if bookings['arrival_date'].min() == pd.Timestamp('2015-07-01') and bookings['arrival_date'].max() == pd.Timestamp('2017-08-31') else "Check the date range",
    },
    {
        "Brief number or claim": "About 1 booking in 4 is cancelled (~25%)",
        "Pandas result": f"{pre_proof_rate:.1%} across train + choose; {hotel_rate_text}",
        "Reading": "Higher than the rough claim; proof-period cancellation labels were not used",
    },
    {
        "Brief number or claim": "Train: before 1 July 2016",
        "Pandas result": f"{len(train):,} bookings; {train['arrival_date'].min():%Y-%m-%d} to {train['arrival_date'].max():%Y-%m-%d}; cancellations {train['is_canceled'].mean():.1%}",
        "Reading": "Check period; used for fitting",
    },
    {
        "Brief number or claim": "Choose: July–December 2016",
        "Pandas result": f"{len(choose):,} bookings; {choose['arrival_date'].min():%Y-%m-%d} to {choose['arrival_date'].max():%Y-%m-%d}; cancellations {choose['is_canceled'].mean():.1%}",
        "Reading": "Check period; use for settings and buffer",
    },
    {
        "Brief number or claim": "Proof: January–August 2017",
        "Pandas result": f"{len(proof):,} bookings; {proof['arrival_date'].min():%Y-%m-%d} to {proof['arrival_date'].max():%Y-%m-%d}",
        "Reading": "Count/date only; cancellation labels held for the final proof run",
    },
    {
        "Brief number or claim": "1 walk per 100 / 10 walks per 1,000",
        "Pandas result": "10 per 1,000 = 1.0% ceiling",
        "Reading": "Same limit arithmetically; not an observed walk rate",
    },
    {
        "Brief number or claim": "EUR 300 per walked guest",
        "Pandas result": "Client-stated scoring assumption",
        "Reading": "Not verifiable from the bookings CSV",
    },
    {
        "Brief number or claim": "Current flat rule: average cancellation rate less a few rooms",
        "Pandas result": "Safety margin is not recorded in the CSV or brief",
        "Reading": "Ask the revenue manager for the current margin",
    },
    {
        "Brief number or claim": "Next 14 nights",
        "Pandas result": "Fourteen-night output requirement",
        "Reading": "MVP has not been run",
    },
    {
        "Brief number or claim": "Vendor demo claimed 95% accuracy",
        "Pandas result": "No demo predictions or evaluation details supplied",
        "Reading": "Unverified; do not use as the acceptance metric",
    },
]

brief_checks = pd.DataFrame(checks)
brief_checks


Brief number or claim,Pandas result,Reading
"119,390 bookings; 32 columns","119,390 rows; 32 columns",Matches the brief
Arrivals July 2015–August 2017,2015-07-01 to 2017-08-31,Matches the stated range
About 1 booking in 4 is cancelled (~25%),36.2% across train + choose; City Hotel: 41.3%; Resort Hotel: 26.3%,Higher than the rough claim; proof-period cancellation labels were not used
Train: before 1 July 2016,"49,157 bookings; 2015-07-01 to 2016-06-30; cancellations 35.8%",Check period; used for fitting
Choose: July–December 2016,"29,546 bookings; 2016-07-01 to 2016-12-31; cancellations 36.9%",Check period; use for settings and buffer
Proof: January–August 2017,"40,687 bookings; 2017-01-01 to 2017-08-31",Count/date only; cancellation labels held for the final proof run
"1 walk per 100 / 10 walks per 1,000","10 per 1,000 = 1.0% ceiling",Same limit arithmetically; not an observed walk rate
EUR 300 per walked guest,Client-stated scoring assumption,Not verifiable from the bookings CSV
Current flat rule: average cancellation rate less a few rooms,Safety margin is not recorded in the CSV or brief,Ask the revenue manager for the current margin
Next 14 nights,Fourteen-night output requirement,MVP has not been run
